**IMPORTS**

In [1]:
!pip install transformers datasets scikit-learn torch -q

In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from tqdm import tqdm

**FILE UPLOAD**

In [43]:
from google.colab import files
uploaded = files.upload()

**MERGING TWO DATA SETS TO CREATE FINAL TRAINING DATASET**

In [44]:
import pandas as pd
import json

csv_path = "classification_dataset.csv"
jsonl_path = "track_classifier_train.jsonl"
output_path = "final_classification_dataset.csv"

df = pd.read_csv(csv_path)

new_rows = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())

        condition = data.get("condition")
        track = data.get("track")

        new_rows.append({
            "input": condition,
            "output": track
        })

new_df = pd.DataFrame(new_rows)

final_df = pd.concat([df, new_df], ignore_index=True)

final_df.to_csv(output_path, index=False)

print("Done!")

Done!


**PREPOCESSING THE VALIDATION DATASET**

In [52]:
import pandas as pd
import json

jsonl_path = "track_classifier_val.jsonl"
output_csv = "final_validation_dataset.csv"

rows = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())

        # Extract values
        sentence = data.get("condition", "").strip()
        track = data.get("track")

        rows.append({
            "input": sentence,
            "output": track
        })

# Create DataFrame
df = pd.DataFrame(rows)

# Save as CSV
df.to_csv(output_csv, index=False)

print("Done! CSV created:", output_csv)

Done! CSV created: final_validation_dataset.csv


**READING TRAINING DATASETS**

In [3]:
df = pd.read_csv("final_classification_dataset.csv")

# Rename columns
df = df.rename(columns={"input": "text", "output": "label"})

# Convert labels (1–5 → 0–4)
df["label"] = df["label"] - 1

df.head()



,text,label
0,was zero or null on the most recent date in th...,2
1,combined data usage and outgoing call MOU over...,0
2,not subscribed to product in last 40 days,1
3,combined account balance and og call revenue o...,0
4,customers who activated HBB add-on yesterday,2


In [4]:
# Add this to the bottom of Cell 4
print("Unique labels in dataset:", df["label"].unique())


Unique labels in dataset: [2 0 1 4 3 5]


In [5]:
# Split the dataset: 80% for training the model, 20% for validating its performance.
# random_state ensures reproducibility (we get the same split every time we run it).

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
# Print the sizes of our splits to confirm.
print("Train size:", len(train_df))
print("Validation size:", len(val_df))

Train size: 3220
Validation size: 805


In [6]:
# Define the base model we want to fine-tune.

model_name = "answerdotai/ModernBERT-base"

# Download and load the Tokenizer associated with this model.
# The tokenizer converts our text sentences into numerical IDs that the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
# Create a custom PyTorch Dataset class to efficiently feed data to our model.
# Custom Dataset class for handling text data and preparing it for model training
# This class inherits from PyTorch's Dataset class and implements required methods:
# - __init__: Initializes the dataset by tokenizing text and preparing labels
# - __getitem__: Returns a single data item with its features and label
# - __len__: Returns the total number of items in the dataset
# The max_length parameter controls the maximum number of tokens per text sample

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
 # Returns the total number of items in the dataset.
    def __len__(self):
        return len(self.texts)

  # Tokenize the text on the fly.
  # It pads short sentences and truncates long ones to ensure a uniform length (max_len).

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

# Return the processed input tensors and the corresponding target label.
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [8]:
import pandas as pd
from torch.utils.data import DataLoader

# 1. Load the Training Data
train_df = pd.read_csv("/content/final_classification_dataset.csv")  # Replace with your train filename
train_df = train_df.rename(columns={"input": "text", "output": "label"})
train_df["label"] = train_df["label"] - 1

# 2. Load the Separate Validation Data
val_df = pd.read_csv("/content/final_validation_dataset.csv")        # Replace with your validation filename
val_df = val_df.rename(columns={"input": "text", "output": "label"})
val_df["label"] = val_df["label"] - 1

print("Train size:", len(train_df))
print("Validation size:", len(val_df))

# 3. Create the Data Loaders (Assuming you already ran Cell 6 and 7 for the Tokenizer and TextDataset)
train_dataset = TextDataset(train_df["text"], train_df["label"], tokenizer)
val_dataset = TextDataset(val_df["text"], val_df["label"], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)


Train size: 4025
Validation size: 350


In [9]:
# Download the pre-trained ModernBERT model.
# We add a classification head on top with num_labels=6 to account for our 6 distinct tracks.

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
# Detect if a GPU (CUDA) is available in Colab for faster training; otherwise fallback to CPU.
# Move the model weights to the selected hardware device.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Using device:", device)

Using device: cuda


In [11]:

# Initialize the AdamW optimizer.
# A learning rate of 2e-5 is standard best practice for fine-tuning transformer models.

optimizer = AdamW(model.parameters(), lr=2e-5)

In [12]:
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")         # Initialize a progress bar for the current epoch.

    for batch in loop:
        optimizer.zero_grad()         # Clear old gradients from the last step.


# Move the batch data to our GPU/CPU.
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
 # Forward pass: get model predictions and calculate the loss (error).
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()
  # Backward pass: calculate the gradients (how much to adjust weights).
        loss.backward()

# Optimizer step: update the model's weights based on the gradients.
        optimizer.step()
 # Update the progress bar with the current loss.
        loop.set_postfix(loss=loss.item())
 # Print the average loss for the epoch to monitor learning progress.
    print(f"Epoch {epoch+1} Loss:", total_loss / len(train_loader))

Epoch 1: 100%|██████████| 252/252 [02:11<00:00,  1.91it/s, loss=0.000559]


Epoch 1 Loss: 0.17471604692377335


Epoch 2: 100%|██████████| 252/252 [02:09<00:00,  1.95it/s, loss=0.000118]


Epoch 2 Loss: 0.0035639114216575677


Epoch 3: 100%|██████████| 252/252 [02:09<00:00,  1.95it/s, loss=0.000126]


Epoch 3 Loss: 0.00039165893198630315


Epoch 4: 100%|██████████| 252/252 [02:10<00:00,  1.93it/s, loss=0.000103]


Epoch 4 Loss: 0.0002250361046554601


Epoch 5: 100%|██████████| 252/252 [02:12<00:00,  1.91it/s, loss=7.65e-5]

Epoch 5 Loss: 0.00014396168688339428


**EVALUATION**

In [14]:
from sklearn.metrics import classification_report                     # Lists to store our predictions and true labels for the classification report.

model.eval()         # Set the model to evaluation mode (disables training features like dropout).
all_preds = []       # Lists to store our predictions and true labels for the classification report.
all_labels = []

with torch.no_grad():   # torch.no_grad() disables gradient tracking to speed up evaluation and save memory.
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

      # Forward pass to get predictions.
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
 # Grab the highest probability class prediction.
        preds = torch.argmax(outputs.logits, dim=1)

        # Store predictions and true labels
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# This will print the Precision, Recall, and F1-Score for EACH of your 5 tracks!
print(classification_report(all_labels, all_preds, target_names=["Track 1", "Track 2", "Track 3", "Track 4", "Track 5", "Track 6"]))


              precision    recall  f1-score   support

     Track 1       0.99      1.00      1.00       240
     Track 2       0.97      1.00      0.99        75
     Track 3       1.00      0.20      0.33         5
     Track 4       1.00      1.00      1.00         5
     Track 5       1.00      1.00      1.00        10
     Track 6       1.00      1.00      1.00        15

    accuracy                           0.99       350
   macro avg       0.99      0.87      0.89       350
weighted avg       0.99      0.99      0.98       350



**PREDICTION**

In [16]:
def predict_multiple(texts):
    # Ensure model is in evaluation mode.
    model.eval()

    # If you accidentally pass a single string instead of a list,
    # this wraps it in a list so the code doesn't break.
    if isinstance(texts, str):
        texts = [texts]

    # Tokenize the entire list of text snippets at once.
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    # Make the predictions for all statements simultaneously.
    with torch.no_grad():
        outputs = model(**inputs)
        # .tolist() converts the PyTorch tensor into a standard Python list
        preds = torch.argmax(outputs.logits, dim=1).tolist()

    # Loop through and print each input with its corresponding Track
    for text, pred in zip(texts, preds):
        track_number = pred + 1  # Add 1 to map 0-indexed prediction back to Track (1 to 6).
        print(f"Input: '{text}'")
        print(f"Track: {track_number}\n")


# --- Test with multiple statements ---
test_statements = [
    "customers who received a bonus for action key X in the last N days",
    "total revenue in the last 30 days",
    "customer network age > 12 months"
]

predict_multiple(test_statements)


Input: 'customers who received a bonus for action key X in the last N days'
Track: 5

Input: 'total revenue in the last 30 days'
Track: 1

Input: 'customer network age > 12 months'
Track: 1



**SAVING MODEL**

In [38]:
# Save the final model weights and the tokenizer configuration locally.
# This allows us to load the model later for production deployment without retraining.

model.save_pretrained("bert_classifier")
tokenizer.save_pretrained("bert_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_classifier/tokenizer_config.json', 'bert_classifier/tokenizer.json')

In [62]:
# model.eval()                  # Set the model to evaluation mode (disables training features like dropout).

# correct = 0
# total = 0

# with torch.no_grad():
#     for batch in val_loader:
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         labels = batch["labels"].to(device)

#         outputs = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask
#         )

#         preds = torch.argmax(outputs.logits, dim=1)

#         correct += (preds == labels).sum().item()
#         total += labels.size(0)

# accuracy = correct / total
# print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.9882352941176471


In [29]:
# # Instantiate our custom dataset for both train and validation sets.

# train_dataset = TextDataset(train_df["text"], train_df["label"], tokenizer)
# val_dataset = TextDataset(val_df["text"], val_df["label"], tokenizer)

# # Wrap datasets in DataLoaders to group data into batches of 16.
# # We shuffle the training data so the model doesn't memorize the order of the inputs.
# train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=16)

In [15]:
# # a helper function to test the model with custom raw text. # Ensure model is in evaluation mode.

# def predict(text):
#     model.eval()

#     inputs = tokenizer(                         # Tokenize the custom text snippet.
#         text,
#         return_tensors="pt",
#         truncation=True,
#         padding=True,
#         max_length=128
#     ).to(device)

#     with torch.no_grad():                            # Make the prediction.
#         outputs = model(**inputs)
#         pred = torch.argmax(outputs.logits, dim=1).item()

#     return pred + 1  # back to original labels,  # Add 1 to map the 0-indexed prediction back to original Track numbers (1 to 5).


# # Test
# print(predict("customers who received a bonus for action key X in the last N days"))

5
